In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

# ===========================
# Potential function classes
# ===========================
class LJ:
    def __init__(self, epsilon=1.0, sigma=1.0):
        self.epsilon = epsilon
        self.sigma = sigma
    def __call__(self, r):
        return 4.0 * self.epsilon * ((self.sigma / r) ** 12 - (self.sigma / r) ** 6)


class PatchyLJ:
    def __init__(self, epsilon=1.0, sigma=1.0, omega=0.262, patch_num=4, epsilon_matrix=None):
        self.epsilon = epsilon
        self.sigma = sigma
        self.omega = omega
        self.LJ_model = LJ(epsilon, sigma)
        self.patch_num = patch_num
        self.epsilon_matrix = epsilon_matrix if epsilon_matrix is not None else np.identity(patch_num)

    def wrap(self, ang):
        return (ang + np.pi) % (2 * np.pi) - np.pi

    def __call__(self, r, phi_i, phi_j):
        pot = 0
        patches = [2*np.pi*n/self.patch_num for n in range(self.patch_num)]
        for i in range(self.patch_num):
            for j in range(self.patch_num):
                theta_i = self.wrap(phi_i + patches[i])
                theta_j = self.wrap(phi_j + patches[j] - np.pi)
                pot += self.LJ_model(r) * self.epsilon_matrix[i,j] * np.exp(-(theta_i**2 + theta_j**2) / (2 * self.omega**2))
        return pot


class OrientedLJ:
    def __init__(self, epsilon=1, sigma=1, m=1, n=2, alpha=np.pi):
        self.m = m
        self.n = n
        self.alpha = alpha
        self.LJ_model = LJ(epsilon, sigma)
        
    def __call__(self, r, phi1, phi2):
        radial = self.LJ_model(r)
        angular = 1 + self.m * np.cos(self.n * (phi2 - phi1) + self.alpha)
        return radial + angular


class MorseWithAngles:
    def __init__(self, De=1.0, re=8.5, a=0.5, C1=0.3, C2=0.1):
        self.De = De
        self.re = re
        self.a = a
        self.C1 = C1
        self.C2 = C2

    def __call__(self, r, phi1, phi2):
        radial = self.De * (np.exp(-2 * self.a * (r - self.re)) - 2 * np.exp(-self.a * (r - self.re)))
        angular = 1 + self.C1 * np.cos(np.radians(phi2 - phi1)) \
                    + self.C2 * np.cos(2 * np.radians(phi2 - phi1))
        return radial * angular


class GeometricLJ:
    def __init__(self, epsilon=1, sigma=1, patch_num=3, epsilon_matrix=None, rho=1, geometric_strength=0.5, S_h=1):
        self.patch_num = patch_num
        self.rho = rho
        self.geometric_strength = geometric_strength
        self.S_h = S_h
        self.LJ_model = LJ(epsilon, sigma)
        self.epsilon_matrix = epsilon_matrix if epsilon_matrix is not None else np.identity(patch_num)
        
    def __call__(self, r, phi1, phi2):
        dx = r
        dy = 0
        total_potential = self.LJ_model(r)
        for i in range(self.patch_num):
            for j in range(self.patch_num):
                theta1 = phi1 + self.S_h* 2*np.pi*i/self.patch_num
                theta2 = phi2 + 2*np.pi*j/self.patch_num
                
                delta_cos = self.rho * (np.cos(theta2) - np.cos(theta1))
                delta_sin = self.rho * (np.sin(theta2) - np.sin(theta1))
                
                r_patch = np.sqrt((dx + delta_cos) * (dx + delta_cos) + (dy + delta_sin) * (dy + delta_sin))
                total_potential += self.epsilon_matrix[i,j] * self.geometric_strength * self.LJ_model(r_patch)
            
        return total_potential
    
    
# ===========================
#       Plotting helper
# ===========================

def plot_potential(potential, r_fixed=9.0, n_points=100):
    phi = np.linspace(0, 2*np.pi, n_points)
    phi1 = np.linspace(0, 2*np.pi, n_points)
    phi2 = np.linspace(0, 2*np.pi, n_points)
    phi1_grid, phi2_grid = np.meshgrid(phi1, phi2)
    
    r_vals = np.linspace(0.9, 3, n_points)

    # 1D plots
    pot_1d_phi = [potential(r_fixed, p, 0) for p in phi]
    pot_1d_r = [potential(r, 0, 0) for r in r_vals]
    fig, ax1 = plt.subplots(nrows=1, ncols=2, figsize=(10, 3))
    plt.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=0.6, hspace=None)

    ax1[0].plot(phi, pot_1d_phi)
    ax1[0].set_xlabel("phi_1")
    ax1[0].set_ylabel("Potential")
    ax1[0].grid(True)
    
    ax1[1].plot(r_vals, pot_1d_r)
    ax1[1].set_xlabel("r")
    ax1[1].set_ylabel("Potential")
    ax1[1].grid(True)

    # 2D plot
    pot_2d = np.array([[potential(r_fixed, p1, p2) for p1, p2 in zip(row1, row2)]
                       for row1, row2 in zip(phi1_grid, phi2_grid)])
    fig, ax2 = plt.subplots(figsize=(4, 3))
    im = ax2.imshow(pot_2d, origin='lower', extent=[0, 2*np.pi, 0, 2*np.pi], aspect='auto')
    ax2.set_xlabel("phi_1")
    ax2.set_ylabel("phi_2")
    fig.colorbar(im, ax=ax2, label="Potential")
    
    plt.tight_layout()
    plt.show()


In [ ]:
matrix1 = np.array ([[.1, 1, 1], [1, .1, 1], [1, 1, .1]])

patches2 = np.array([[1.0, 0.5, 0.1, 0.0], [0.5, 1.0, 0.5, 0.1], [0.1, 0.5, 1.0, 0.5], [0.0, 0.1, 0.5, 1.0]])

n=18

# pot = MorseWithAngles(De=1.0, re=8.5, a=0.5, C1=0.3, C2=0.1)
# pot = OrientedLJ()
pot = PatchyLJ(epsilon=1.0, sigma=1.0, patch_num=n
            #    , epsilon_matrix=np.ones((n, n))
               )

# pot = GeometricLJ(patch_num=n, S_h=1)

plot_potential(pot, r_fixed=7)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

CONFIG = {
    'energy_ref': -6227.1749,
    'zeta_max': 1.39687500,
    'phi2_max': 20
}

def process_data(filename):
    df = pd.read_csv(filename, sep=r'\s+', header=None,
                     names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    df['energy'] -= 2 * CONFIG['energy_ref']
    return df

SSdata_all = process_data(data_path)

In [ ]:
SSdata_all.drop_duplicates(subset=['phi1', 'phi2', 'r']).energy.unique()

In [ ]:
reducedData = SSdata_all.drop_duplicates(subset=['phi1', 'phi2', 'r']).drop(columns='zeta')
# copied1 = reducedData[reducedData.phi1 ==0]
# copied2 = reducedData[reducedData.phi1 ==0]
# copied2.phi2 = copied1['phi2'] + 20
# # copied._append(cp2, axis = 1)
# copied1._append(copied2).phi1.min

In [ ]:
reducedData

In [ ]:
import pandas as pd
import numpy as np
import time

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

CONFIG = {
    'energy_ref': -6227.1749
}

def process_data(filename):
    start_time = time.time()
    df = pd.read_csv(filename, sep=r'\s+', header=None,
                     names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    
    df['energy'] -= 2 * CONFIG['energy_ref']
    
    print(f"Data loaded in {time.time() - start_time:.2f} seconds")
    print(f"Original dataset shape: {df.shape}")
    
    return df

def remove_zeta_dimension(df):
    start_time = time.time()
    df_reduced = df.drop_duplicates(subset=['phi1', 'phi2', 'r'])
    
    print(f"Zeta dimension removed in {time.time() - start_time:.2f} seconds")
    print(f"Reduced dataset shape: {df_reduced.shape}")
    
    return df_reduced

def prepare_for_periodic_fitting(df):
    df_periodic = pd.DataFrame(columns==['phi1', 'phi2', 'zeta', 'r', 'energy'])
    for i in range(df.phi1.min, df.phi2.max):
        copied = reducedData[reducedData.phi1 ==i]
        copied.phi2 = copied1['phi2'] + 20
        df_periodic._append(copied2)
            
    return df_periodic


data_all = process_data(data_path)
data_reduced = remove_zeta_dimension(data_all)
data_final = data_reduced.drop(columns=['zeta'])
data_periodic = prepare_for_periodic_fitting(data_final)

data_periodic

In [ ]:
import pandas as pd
import numpy as np
import time

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

CONFIG = {
    'energy_ref': -6227.1749
}

def process_data(filename):
    df = pd.read_csv(filename, sep=r'\s+', header=None, names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    
    df['energy'] -= 2 * CONFIG['energy_ref']    
    return df

def remove_zeta_dimension(df):
    df_reduced = df.drop_duplicates(subset=['phi1', 'phi2', 'r'])
    return df_reduced

def extend_phi2_range(df):    
    frames = []
    num_repeats = 18
    
    for i in range(num_repeats):
        segment_df = df.copy()
        
        segment_df['phi2'] = segment_df['phi2'] + (i * 20)
        
        frames.append(segment_df)
    
    df_extended = pd.concat(frames, ignore_index=True)
    
    return df_extended


data_all = process_data(data_path)
data_reduced = remove_zeta_dimension(data_all)
data_final = data_reduced.drop(columns=['zeta'])
data_extended = extend_phi2_range(data_final)

In [ ]:
grid_df = data_extended[data_extended.r == 8.2].pivot_table(index='phi2', columns='phi1', values='energy')
Phi1, Phi2 = np.meshgrid(grid_df.columns, grid_df.index)
E_grid = grid_df.values

fig, ax = plt.subplots(figsize=(6,4))
c0 = ax.contourf(Phi1, Phi2, E_grid, levels=100, cmap='viridis')
ax.set_title("DFTB Energy Map")
ax.set_xlabel("phi1 (deg)")
ax.set_ylabel("phi2 (deg)")
fig.colorbar(c0, ax=ax, shrink=1)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import time

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

CONFIG = {
    'energy_ref': -6227.1749
}

def process_data(filename):
    """Load and preprocess data."""
    start_time = time.time()
    df = pd.read_csv(filename, sep=r'\s+', header=None,
                     names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    
    df['energy'] -= 2 * CONFIG['energy_ref']
    
    print(f"Data loaded in {time.time() - start_time:.2f} seconds")
    print(f"Original dataset shape: {df.shape}")
    
    return df

def remove_zeta_dimension(df):
    """Remove the zeta dimension by keeping only one row per unique combination."""
    start_time = time.time()
    df_reduced = df.drop_duplicates(subset=['phi1', 'phi2', 'r'])
    
    print(f"Zeta dimension removed in {time.time() - start_time:.2f} seconds")
    print(f"Reduced dataset shape: {df_reduced.shape}")
    
    return df_reduced

def apply_periodic_boundary_conditions(df, screw_direction=1):
    """
    Apply periodic boundary conditions similar to the supervisor's implementation.
    
    This function will create a DataFrame with periodic boundary conditions 
    applied to phi2, making it repeat with period 20.
    
    Args:
        df: DataFrame with phi1, phi2, r, energy columns
        screw_direction: Direction of the screw symmetry (1 or -1)
        
    Returns:
        DataFrame with periodic boundary conditions applied
    """
    print("Applying periodic boundary conditions...")
    start_time = time.time()
    
    # First, ensure the data is sorted by phi1, phi2, and r
    df_sorted = df.sort_values(by=['phi1', 'phi2', 'r']).reset_index(drop=True)
    
    # Get unique phi1 and r values
    unique_phi1 = df_sorted['phi1'].unique()
    unique_r = df_sorted['r'].unique()
    
    # Create a new list to hold all data frames
    all_dfs = []
    
    # For each unique phi1 and r combination
    for phi1_val in unique_phi1:
        for r_val in unique_r:
            # Get the subset of data for this phi1, r combination
            subset = df_sorted[(df_sorted['phi1'] == phi1_val) & (df_sorted['r'] == r_val)]
            
            # Skip if no data for this combination
            if len(subset) == 0:
                continue
                
            # The period of phi2 is 20
            period = 20
            
            # Create 18 copies to cover 0-359 degrees
            for i in range(18):
                # Create a copy of the subset
                subset_copy = subset.copy()
                
                # Apply the shift to phi2 based on screw_direction
                subset_copy['phi2'] = subset_copy['phi2'] + (i * period * screw_direction)
                
                # Add to the list of dataframes
                all_dfs.append(subset_copy)
    
    # Concatenate all dataframes
    df_periodic = pd.concat(all_dfs, ignore_index=True)
    
    print(f"Periodic boundary conditions applied in {time.time() - start_time:.2f} seconds")
    print(f"Extended dataset shape: {df_periodic.shape}")
    
    # Verify the phi2 range
    print(f"Phi2 range: {df_periodic['phi2'].min()} to {df_periodic['phi2'].max()}")
    print(f"Number of unique phi2 values: {df_periodic['phi2'].nunique()}")
    
    return df_periodic

data_all = process_data(data_path)
data_reduced = remove_zeta_dimension(data_all)
data_final = data_reduced.drop(columns=['zeta'])
data_periodic = apply_periodic_boundary_conditions(data_final)

In [ ]:
import matplotlib.pyplot as plt
def plot_pr(data):
    grid_df = data[data.r == 8.2].pivot_table(index='phi2', columns='phi1', values='energy')
    Phi1, Phi2 = np.meshgrid(grid_df.columns, grid_df.index)
    E_grid = grid_df.values

    fig, ax = plt.subplots(figsize=(6,4))
    c0 = ax.contourf(Phi1, Phi2, E_grid, levels=100, cmap='viridis')
    ax.set_title("DFTB Energy Map")
    ax.set_xlabel("phi1 (deg)")
    ax.set_ylabel("phi2 (deg)")
    fig.colorbar(c0, ax=ax, shrink=1)
    plt.show()
    
plot_pr(data_periodic)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

interaction_type = 'SS'
data_path = f'/home/hadis/selfassembly/Data_Figures_Paper/E_all_{interaction_type}.dat'

CONFIG = {
    'energy_ref': -6227.1749,
    'zeta_max': 1.39687500,
    'phi2_max': 20
}

def process_data(filename):
    df = pd.read_csv(filename, sep=r'\s+', header=None,
                     names=['phi1', 'phi2', 'zeta', 'r', 'energy'])
    df['energy'] -= 2 * CONFIG['energy_ref']
    df['zeta'] = np.where(
        df['zeta'] < CONFIG['zeta_max'] / 2,
        df['zeta'],
        df['zeta'] - CONFIG['zeta_max']
    )
    return df

SSdata_all = process_data(data_path)
# data = SSdata_all.drop(columns=['zeta', 'r'])

# for i in range (20):
#     print(i)
#     replicat_rows = data[data.phi1 == 0].copy()
#     replicat_rows.loc[:, 'phi2'] += 20

#     data = data._append(replicat_rows, ignore_index=True)
# data

In [ ]:
SSdata_all

In [ ]:
SSdata_all['zeta'][1]!=SSdata_all['zeta'][0]

In [ ]:
for i in range(20):
# for i in range(SSdata_all.index[-1]):
    if (SSdata_all['zeta'][i]!=SSdata_all['zeta'][i+1]):
        select = SSdata_all.loc[SSdata_all['zeta']==SSdata_all['zeta'][i], :]
select

In [ ]:
# for i in range(20):
for i in range(SSdata_all.index[-1]):
    remove = SSdata_all.loc[(SSdata_all['zeta']==SSdata_all['zeta'][i]) & (SSdata_all['r']==SSdata_all['r'][i]), :].index

    dropted = SSdata_all.drop(index=remove[1:])
    if i%100000==0:
        print(i)

In [ ]:
# data2 = np.where(data['r']==8)

# --- Pivot for plotting ---
grid_df = data2.pivot_table(index='phi2', columns='phi1', values='energy')
Phi1, Phi2 = np.meshgrid(grid_df.columns, grid_df.index)
E_grid = grid_df.values

# --- Plot ---
fig, ax = plt.subplots(figsize=(6,4))
c0 = ax.contourf(Phi1, Phi2, E_grid, levels=100, cmap='viridis')
ax.set_title("DFTB Energy Map")
ax.set_xlabel("phi1 (deg)")
ax.set_ylabel("phi2 (deg)")
fig.colorbar(c0, ax=ax, shrink=1)
plt.show()

In [ ]:
SSdata_all.head(5)
SSdata_all.tail(10)
SSdata_all.columns
SSdata_all.T
SSdata_all.sort_values(by='phi2', ascending=False)

for i in range(20):
    selected = SSdata_all[SSdata_all['phi1']==i]
     
selected_data = [SSdata_all[SSdata_all['phi1'] == i] for i in range(20)]

In [ ]:

# --- Process and extend data ---
SSdata_all = process_data(data_path)
SSdata = np.where(SSdata_all['r']==8)
# SSdata = screw_pbc(get_min_energy_per_phi(SSdata))
# SSdata_full = screw_pbc(SSdata, screw_direction=1)

# --- Pivot for plotting ---
grid_df = SSdata.pivot_table(index='phi2', columns='phi1', values='energy')
Phi1, Phi2 = np.meshgrid(grid_df.columns, grid_df.index)
E_grid = grid_df.values

# --- Plot ---
fig, ax = plt.subplots(figsize=(6,4))
c0 = ax.contourf(Phi1, Phi2, E_grid, levels=100, cmap='viridis')
ax.set_title("DFTB Energy Map")
ax.set_xlabel("phi1 (deg)")
ax.set_ylabel("phi2 (deg)")
fig.colorbar(c0, ax=ax, shrink=1)
plt.show()